# Using the API

This notebook shows how to query the prediction API.

**Before you start**, in a terminal:

```bash
make run_api        # starts the API on http://127.0.0.1:8000
```

The API needs no model to start: `/predict` will answer `503` until
`make run_train` has produced a model (or until `PUT /model` has loaded one).

The interactive documentation lives at <http://127.0.0.1:8000/docs>.

In [ ]:
import requests

BASE_URL = "http://127.0.0.1:8000"

## 1. Service health

In [ ]:
response = requests.get(f"{BASE_URL}/")
print(response.status_code, response.json())

## 2. Model state

Lets you check that a deployment did load a model without triggering a
prediction — and get the cause of the failure if it did not.

In [ ]:
response = requests.get(f"{BASE_URL}/model")
print(response.status_code, response.json())

## 3. Single prediction

The parameters are validated by Pydantic: a nonsensical value (negative
distance, hour outside 0-23) or a missing field receives an explicit `422`
BEFORE it ever reaches the model.

In [ ]:
params = {
    "distance_km": 5.0,
    "passengers": 2,
    "hour": 14,
    "day_of_week": "monday",
}

response = requests.get(f"{BASE_URL}/predict", params=params)
print(response.status_code, response.json())

### Error case: invalid input

In [ ]:
response = requests.get(f"{BASE_URL}/predict", params={**params, "distance_km": -1})
print(response.status_code, response.json())

## 4. Batch prediction

The body is expected to be a **list** of objects, and the response keeps the
order of the inputs.

In [ ]:
payload = [
    {"distance_km": 5.0, "passengers": 2, "hour": 14, "day_of_week": "monday"},
    {"distance_km": 12.5, "passengers": 1, "hour": 23, "day_of_week": "saturday"},
]

response = requests.post(f"{BASE_URL}/predict_batch", json=payload)
print(response.status_code, response.json())

## 5. Command-line equivalent

⚠️ Inside **single** quotes, IPython does NOT interpolate `{...}` variables:
the command used to send the literal string `{payload_str}` to the server. So
the JSON is written to a file and handed to `curl` through
`--data-binary @file`, which avoids any interpolation inside a shell string.

In [ ]:
import json
from pathlib import Path

# *.json is git-ignored: this temporary file does not pollute the repository.
Path("payload.json").write_text(json.dumps(payload))

In [ ]:
!curl -s -X POST "{BASE_URL}/predict_batch" \
  -H "Content-Type: application/json" \
  --data-binary @payload.json